<table>
  <tr>
    <td><div align="left"><font size="30">Block-diagram simulation</font></div></td>
    <td><img src="support/figs/BDSimLogo_NoBackgnd@2x.png" width="300"></td>
  </tr>
</table>

<p></p>
<table align="center" style="border: none; border-collapse: collapse;">
  <tr>
    <td style="border: none; font-size: 1.5em; padding: 2px 8px;">🧠</td>
    <td style="border: none; font-size: 1.5em; padding: 2px 8px;">Block diagram thinking</td>
  </tr>
  <tr>
    <td style="border: none; font-size: 1.5em; padding: 2px 8px;">🐍</td>
    <td style="border: none; font-size: 1.5em; padding: 2px 8px;">Python coding</td>
  </tr>
</table>

(c) Peter Corke 2026

In [ ]:
import importlib
import setup_tutorial
importlib.reload(setup_tutorial)
await setup_tutorial.setup_tutorial(required_toolboxes=['bdsim', 'roboticstoolbox'], required_packages=['matplotlib', 'numpy', 'scipy'])

print("\nRunning bdsim version", importlib.import_module("bdsim").__version__)

In [ ]:
import bdsim
import matplotlib.pyplot as plt

# A first-order system




Here's a hand drawn sketch of a simple first-order system with proportional feedback

![Block diagram sketch](support/figs/bd1-sketch.png)

We model it using `bdsim` by writing Python code to represent the model.  Python means you can use your
favourite IDE, debugger, and version control.

In [ ]:
sim = bdsim.BDSim(animation=True)  # create simulator
bd = sim.blockdiagram()  # create an empty block diagram

# define the blocks
demand = bd.STEP(T=1, name="demand")
sum = bd.SUM("+-")
gain = bd.GAIN(10)
plant = bd.LTI_SISO(0.5, [2, 1], name="plant")
scope = bd.SCOPE(styles=["k", "r--"], loc="lower right")

# connect the blocks
bd.connect(demand, sum[0], scope[1])
bd.connect(plant, sum[1])
bd.connect(sum, gain)
bd.connect(gain, plant)
bd.connect(plant, scope[0])

Before we can run the code, we need to compile it. That does a bunch of checks and builds the datastructures required for execution.

In [ ]:
bd.compile(verbose=True)  # check the diagram

and this is a succinct representation of the model, showing each block and where it's inputs are sourced from.

In [ ]:

bd.report_summary()

Now we can run the model.  In this case for 5 seconds, and we'll see the output of the `SCOPE` block.  We will also record the outputs of the `demand` and `sum` blocks.

In [ ]:
out = sim.run(bd, T=5, watch=[demand, sum])  # simulate for 5s

In [ ]:
print(out)

In [ ]:
from IPython.display import Markdown, display

mermaid = bd.graph(format="mermaid") # export a graph in Mermaid format
display(Markdown(f"``` mermaid\n{mermaid}\n```")) # Render dynamically via Jupyter's built-in Mermaid engine

# The "bouncing ball" problem



This is a classic hybrid simulation problem with continuous dynamics (the ball flight) and discrete dynamics (the impact). 

In [ ]:
import bdsim

# define constants
g = -9.81  # gravity m/s2
e = 0.8  # coefficient of restitution
h0 = 10  # initial height m

sim = bdsim.BDSim(animation=True)  # create simulator
bd = sim.blockdiagram()  # create an empty block diagram

# define the blocks
gravity = bd.CONSTANT(g, name="gravity")

# chain of integrators for ball flight dynamics
velocity = bd.INTEGRATOR(name="velocity", x0=0)  # initial height is initial velocity
position = bd.INTEGRATOR(name="position", x0=h0)

scope = bd.SCOPE(styles=["k", "r--"], loc="lower right")

# ---------------- the impact event detection and response -------------------- #
# detect when the ball hits the ground (position=0) and trigger an event
def impact(event_block, state_map):
    state_map[velocity] *= -e  # reverse velocity and apply restitution

# connect the impact function to the event triggered when position crosses zero from above
ground = bd.EVENT("-", impact)

# ----------------------------------------------------------------------------- #

# connect the blocks together
bd.connect(gravity, velocity)
bd.connect(velocity, position)
bd.connect(velocity, scope[0])
bd.connect(position, scope[1])
bd.connect(position, ground)

bd.compile()  # check the diagram
bd.report_summary()

In [ ]:
out = sim.run(bd, T=5)  # , watch=[demand, sum])  # simulate for 5s

# Cart-pole system



This is another classical simulation problem, a non-linear system comprising two coupled dynamical systems.

In [ ]:
%run support/cartpole.py -q 

# Quadrotor flight controller


This block diagram is a complete controller for a quadrotor and drives it to follow a circular path.  The controller has nested loops, from the inside going outwards:

* roll/pitch attitude control
* CoM velocity control
* CoM position control

and a separate yaw controller, independent of the CoM motion

![quadrotor block diagram model](support/figs/fig4_24.png)

You can open the file `quadrotor.py` to see the model, it's quite large.

In [ ]:
%run support/quadrotor.py -q --tiles 2x2

# The block diagram editor



`bdsim` includes a Qt-based editor called `bdedit`.  It provides a decent block diagram drawing environment and the results are saved in JSON files with a `.bd` extension.

![complex](support/figs/computed-torque.png)



These diagrams can be parsed into block diagram objects which can then be simulated, as in the examples above.

In [ ]:
from bdsim import BDSim, bdload
from roboticstoolbox.models.DH import Puma560

robot = Puma560().nofriction()

sim = BDSim(animation=True, quiet=True)
bd = sim.blockdiagram()
clock_fb = bd.clock(50, unit="Hz")

bd = bdload(bd, "support/computed-torque.bd", globalvars=globals())
bd.compile()

sim.report(bd, "summary")
out = sim.run(bd, 5)
print(out)
